# Reading BigQuery Iceberg tables from Spark

Run this in a **BigQuery Studio** notebook (BigQuery > Notebooks > Create notebook).
It uses Serverless for Apache Spark through the `DataprocSparkSession` builder,
so there is no cluster to create.

Work through `ICEBERG_TUTORIAL.md` first: the tables this notebook reads are
created there.

Spark reaches these tables three ways, and the notebook covers all three:

1. **BigQuery Storage API** (`.format("bigquery")`) for managed tables. Live,
   read-write, no setup. This is the one to use.
2. **Iceberg libraries** (`.format("iceberg")`) reading files by path. Read-only
   and only as current as the last `EXPORT TABLE METADATA`. Shown so you can see
   what the export is for.
3. **Iceberg REST catalog** for Lakehouse runtime catalog tables. Live and
   read-write over an open protocol, no Google-specific connector needed.

In [ ]:
PROJECT = "my-project"
BUCKET = "my-bucket"
CATALOG = "iceberg_demo"
NAMESPACE = "sales"

# Iceberg is not bundled with the Spark runtime; pin a version explicitly.
ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.1"

## 1. Start the Spark session

The two catalog properties below register the Lakehouse runtime catalog as a
Spark catalog named `lakehouse`, pointing at the Iceberg REST endpoint. That is
what lets Spark and BigQuery see the same tables.

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

session = Session()
props = session.runtime_config.properties

props["spark.jars.packages"] = ICEBERG_PACKAGE
props["spark.sql.extensions"] = (
    "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
)

# Register the Lakehouse runtime catalog as a REST catalog named 'lakehouse'.
c = "spark.sql.catalog.lakehouse"
props[c] = "org.apache.iceberg.spark.SparkCatalog"
props[f"{c}.type"] = "rest"
props[f"{c}.uri"] = "https://biglake.googleapis.com/iceberg/v1/restcatalog"
props[f"{c}.warehouse"] = f"bl://projects/{PROJECT}/catalogs/{CATALOG}"
props[f"{c}.header.x-goog-user-project"] = PROJECT
props[f"{c}.rest.auth.type"] = "org.apache.iceberg.gcp.auth.GoogleAuthManager"
props[f"{c}.io-impl"] = "org.apache.iceberg.gcp.gcs.GCSFileIO"

spark = (
    DataprocSparkSession.builder
    .appName("iceberg-tutorial")
    .dataprocSessionConfig(session)
    .getOrCreate()
)
spark

## 2. Part 1: managed tables, the practical way

Spark can reach a managed Iceberg table two ways, and the difference matters more
than anything else in this notebook.

| | BigQuery Storage API | Iceberg libraries |
|---|---|---|
| Spark format | `.format("bigquery")` | `.format("iceberg")` |
| Reaches the table by | gRPC to BigQuery | reading files in the bucket |
| Needs `EXPORT TABLE METADATA` | no | **yes** |
| Sees | live table state | the last exported snapshot |
| Spark can write | **yes** | no |

**Use the Storage API.** It is live, it is read-write, and there is no export step
to forget. The section after this one shows the file-based path so you can see why
it exists, not because you should reach for it first.

In [ ]:
# Read the managed table over the Storage Read API. No export needed: this is
# BigQuery's live state, exactly what the console would show.
df = spark.read.format("bigquery").load(f"{PROJECT}.iceberg_lab.orders_managed")
df.show(truncate=False)
print("row count:", df.count())

In [ ]:
# Write to it. writeMethod=direct uses the Storage Write API (gRPC) instead of
# staging files through GCS, so nothing is copied and no export is involved.
import datetime
from pyspark.sql import Row

new_rows = spark.createDataFrame([
    Row(order_id=10, customer="spark-co", amount=42.50,
        order_date=datetime.date(2026, 2, 1)),
])

new_rows.write.format("bigquery") \
    .option("writeMethod", "direct") \
    .mode("append") \
    .save(f"{PROJECT}.iceberg_lab.orders_managed")

# Read it straight back. The row is there immediately, with no export step.
spark.read.format("bigquery") \
    .load(f"{PROJECT}.iceberg_lab.orders_managed") \
    .orderBy("order_id").show(truncate=False)

Go check the BigQuery console: `order_id` 10 is there. Spark wrote to a managed
Iceberg table, and BigQuery saw it immediately.

### The file-based path, and why it exists

The cell below reads the same table the other way, straight from the Iceberg files
in the bucket. This is what an engine does when it speaks Iceberg but has no
BigQuery connector, and it is the arrangement that makes `EXPORT TABLE METADATA`
necessary.

It works, but note what you give up: it is read-only, and it shows the last
exported snapshot rather than the live table. If you have not run
`EXPORT TABLE METADATA` since creating the table, it fails outright with
`Cannot parse missing int: format-version`. If you ran the export before the write
cell above, it will succeed but will be missing `order_id` 10.

That gap is the whole argument for the Storage API, and for Part 2's catalog.

In [ ]:
# The same table, read from the exported Iceberg metadata in the bucket.
# No catalog server is involved: Spark reads version-hint.text, then the
# metadata JSON, then the manifests, then the Parquet files.
managed_path = f"gs://{BUCKET}/iceberg_lab/orders_managed"

try:
    snap = spark.read.format("iceberg").load(managed_path)
    snap.orderBy("order_id").show(truncate=False)
    print("snapshot row count:", snap.count())
    print("compare with the live count above")
except Exception as e:
    print("failed:", type(e).__name__, str(e)[:200])
    print()
    print("Run this in the BigQuery console, then retry:")
    print(f"  EXPORT TABLE METADATA FROM `{PROJECT}.iceberg_lab.orders_managed`;")

**The point:** two routes to the same table. The Storage API talks to BigQuery and
sees the live table, in both directions. The Iceberg-libraries route reads files in
a bucket and sees whatever the last export wrote, in one direction only.

Reach for the Storage API whenever a BigQuery connector exists for your engine.
The file-based route is the fallback for engines that only speak Iceberg, and its
staleness is exactly the problem the Lakehouse catalog solves next.

## 3. Part 2: read and write catalog tables

Here Spark uses the shared catalog by name instead of by path, so no export step
is involved and no snapshot goes stale. Spark writes are visible to BigQuery.

In [ ]:
spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)
spark.sql(f"SHOW TABLES IN lakehouse.{NAMESPACE}").show(truncate=False)

In [ ]:
# Read the table the tutorial created from BigQuery.
spark.sql(f"SELECT * FROM lakehouse.{NAMESPACE}.orders ORDER BY id").show()

In [ ]:
# Write from Spark, then query the same rows from BigQuery to prove the
# catalog is genuinely shared.
spark.sql(
    f"INSERT INTO lakehouse.{NAMESPACE}.orders VALUES (3, 'written-by-spark')"
)
spark.sql(f"SELECT * FROM lakehouse.{NAMESPACE}.orders ORDER BY id").show()

In [ ]:
# Create a table from Spark. It should appear in BigQuery under the same
# four-part name: PROJECT.CATALOG.NAMESPACE.TABLE
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS lakehouse.{NAMESPACE}.from_spark (
        id BIGINT,
        note STRING
    ) USING ICEBERG
""")
spark.sql(f"INSERT INTO lakehouse.{NAMESPACE}.from_spark VALUES (1, 'hello from spark')")
spark.sql(f"SELECT * FROM lakehouse.{NAMESPACE}.from_spark").show(truncate=False)

Now go back to BigQuery and run:

```sql
SELECT * FROM `PROJECT_ID.iceberg_demo.sales.from_spark`;
```

A table created in Spark, queried in BigQuery, with no export and no copy.

## 4. Part 3: external tables

Little is new for Spark here. A BigQuery external table is just an Iceberg table
in a bucket that someone else wrote, which is the same file-based read shown in
section 2. The difference is entirely on the BigQuery side: BigQuery treats it as
read-only and pinned to one metadata file.

The interesting direction is the reverse one. Write an Iceberg table from Spark,
then point a BigQuery external table at it, which is the realistic case: another
engine owns the data and BigQuery queries it.

In [ ]:
ext_path = f"gs://{BUCKET}/iceberg_lab/spark_written"

spark.createDataFrame(
    [(1, "spark-row-one"), (2, "spark-row-two")],
    "id BIGINT, note STRING",
).writeTo(f"lakehouse.{NAMESPACE}.spark_written").using("iceberg").createOrReplace()

spark.sql(f"SELECT * FROM lakehouse.{NAMESPACE}.spark_written").show(truncate=False)

In [ ]:
# Find the metadata file to point a BigQuery external table at.
meta = spark.sql(
    f"SELECT file FROM lakehouse.{NAMESPACE}.spark_written.metadata_log_entries "
    "ORDER BY timestamp DESC LIMIT 1"
).collect()[0]["file"]
print("latest metadata file:")
print(meta)
print()
print("Use it in BigQuery:")
print(f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT}.iceberg_lab.spark_written_ext`
WITH CONNECTION `{PROJECT}.us.big_lake_demo`
OPTIONS (format = 'ICEBERG', uris = ['{meta}']);
""")

## 5. Shut down

Sessions also stop on their own after the configured TTL.

In [ ]:
spark.stop()